In [1]:
#------------------------------------------------ Begin_Librairie ----------------------------------------
import os
import re
import time
import datetime
import urllib3
import requests
import pandas as pd
from bs4 import BeautifulSoup

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)


In [2]:
#------------------------------------------------ Begin_fileName ----------------------------------------
regulatorName = 'GB IMFSC'

print(f"Running {regulatorName} Web Scraping Tool v.1.0")

now = datetime.datetime.now()
processdate = now.strftime('%Y-%m-%d')
filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":", ".")[:-7])

# ------ At first we will define the workspace path -----
try:
    scriptfolder = os.path.dirname(os.path.abspath(__file__))  ## production environment (.py)
except NameError:
    scriptfolder = os.getcwd()  ## notebook environment

os.chdir(scriptfolder)

tempfolder = os.path.join(scriptfolder, 'tempfolder')
if os.path.exists(tempfolder):
    for rem in os.listdir(tempfolder):
        os.remove(os.path.join(tempfolder, rem))
else:
    os.mkdir(tempfolder)


Running GB IMFSC Web Scraping Tool v.1.0


In [3]:
#------------------------------------------------ Begin_session ----------------------------------------
# The IOMFSA register is fully server-rendered - no JS, no cookies, no auth.
# verify=False is required behind the corporate TLS proxy.
BASE = 'https://www.iomfsa.im'

HEADERS = {
    'User-Agent': ('Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 '
                   '(KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36'),
    'Accept-Language': 'en-GB,en;q=0.9',
}

session = requests.Session()
session.headers.update(HEADERS)
session.verify = False


def get_soup(url, timeout=60, retries=3):
    for attempt in range(retries):
        try:
            resp = session.get(url, timeout=timeout)
            resp.raise_for_status()
            return BeautifulSoup(resp.text, 'html.parser')
        except Exception as exc:
            if attempt == retries - 1:
                raise
            print(f'[WARN] : - retry {attempt+1}/{retries} on {url} ({type(exc).__name__})')
            time.sleep(2)


In [4]:
# NOTE : BusinessType must be repeated as a scalar (BusinessType=1&BusinessType=2&...).
# The bracketed form BusinessType[]= is silently ignored by the site and returns a
# different, much smaller result set.
BUSINESS_TYPES = {
    1: 'Authorised Insurer', 2: 'Bank Representative Office', 3: 'Collective Investment Scheme',
    4: 'Corporate Services', 5: 'Credit Unions', 6: 'Crowdfunding Platforms', 7: 'Deposit Taking',
    8: 'Designated Business', 9: 'General Insurance Business Intermediary', 10: 'Insurance Groups',
    11: 'Insurance Manager', 12: 'Insurance Permit Holder', 13: 'Investment Business',
    14: 'Management or Administration to Licenceholders', 15: 'Money Transmission Services',
    16: 'Professional Schemes Administrator', 17: 'Professional Officers',
    18: 'Restricted Deposit Taking', 19: 'Services to Collective Investment Schemes', 20: 'Trust Services',
}

# entity-current=on keeps current entities only (i.e. "uncheck Former entity" from the ticket).
RESULTS_URL = (BASE + '/register-results?entity-name=&entity-current=on&'
               + '&'.join(f'BusinessType={i}' for i in sorted(BUSINESS_TYPES)))

regdict = {
    'GB IMFSC 1': RESULTS_URL,
}

sqldict = {'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [],
          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [],
          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [],
          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [],
          'Phone - Mother company': []}

Typology = {
    'GB IMFSC 1': 'All entities currently regulated by the Isle of Man Financial Services Authority',
}

# ListLabel : 1 = bank, 2 = insurance, 3 = bank & insurance, 4 = everything else.
# Single combined register. It is majority non-bank / non-insurance (designated business,
# corporate & trust services, collective investment schemes), but it does carry both
# deposit-takers (Deposit Taking / Restricted Deposit Taking / Bank Representative Office /
# Credit Unions) and insurers (Authorised Insurer / Insurance Manager / Insurance Permit
# Holder / Insurance Groups), so it is labelled 3 rather than 4 - labelling it 4 would drop
# those entities from both the bank and the insurance feed.
ListLabel = {
    'GB IMFSC 1': 3,
}

# Detail-page field labels vary by register family, so map by <th> text, never by position.
ADDRESS_LABELS = ['Registered Office', 'Principal Trading Address', 'Address', 'Place of Business']
ID_LABELS = ['Reference Number', 'Licence Number', 'FSA Ref.']
REGDATE_LABELS = ['Date Registered', 'Date of Registration']
CEASED_LABELS = ['Date Authorisation/Registration Ceased']
LICENCE_LABELS = ['Designated Business Category(ies)', 'Scheme Type',
                  'Classes/Categories Of Regulated Activity', 'Class(es) of Regulated Activity']

empty_ = ''


In [5]:
#------------------------------------------------ Begin_Fonction ----------------------------------------
def bourange_same_length_array(sqldict):
    maxlen = len(sqldict['ListProcessDate'])
    for key, val in sqldict.items():
        if len(sqldict[key]) != maxlen:
            empty = []
            total_empty = maxlen - len(sqldict[key])
            for i in range(total_empty):
                empty.append('')
            sqldict[key] = sqldict[key] + empty
    return sqldict


def clean(text):
    return ' '.join(text.split())


def flatten_address(text):
    """Detail addresses use \\r\\n line breaks; collapse to a single comma-separated line."""
    parts = [p.strip().rstrip(',') for p in re.split(r'[\r\n]+', text) if p.strip()]
    return ', '.join(parts)


ZIP_RE = re.compile(r'\b(IM\d{1,2}\s*\d[A-Z]{2})\b', re.I)


def parse_zip(address):
    m = ZIP_RE.search(address or '')
    return m.group(1).upper() if m else ''


# Many detail addresses are a single unpunctuated line ("Ragnall House 18 Peel Road Douglas
# IM1 4LZ"), so splitting on commas alone picks the street. Match a known locality first.
IOM_TOWNS = [
    'Port St Mary', 'Port Erin', 'Kirk Michael', 'Union Mills', 'Glen Maye', 'Glen Vine',
    'St Johns', "St John's", 'Union Mills', 'Douglas', 'Onchan', 'Ramsey', 'Peel', 'Castletown',
    'Ballasalla', 'Ballabeg', 'Ballaugh', 'Baldrine', 'Andreas', 'Braddan', 'Bride', 'Colby',
    'Crosby', 'Dalby', 'Foxdale', 'Jurby', 'Laxey', 'Lezayre', 'Lonan', 'Malew', 'Marown',
    'Maughold', 'Michael', 'Patrick', 'Rushen', 'Santon', 'Sulby', 'Strang', 'Sandygate',
    'Cregneash', 'Orrisdale', 'Ronague', 'Dhoon', 'Injebreck',
]
TOWN_RES = [(t, re.compile(r'\b' + re.escape(t) + r'\b', re.I)) for t in IOM_TOWNS]

STREET_WORDS = ('road', 'street', 'floor', 'house', 'court', 'avenue', 'lane', 'way', 'park',
                'drive', 'buildings', 'quay', 'terrace', 'square', 'po box', 'hill', 'close')


def parse_city(address):
    """'<street>, <town>, Isle of Man, IM1 1AE' -> town. Falls back to a comma split."""
    if not address:
        return ''

    # 1) known Isle of Man locality anywhere in the string - take the LAST occurrence,
    #    since street names can also contain a town name ("18 Peel Road ... Douglas").
    best, best_pos = '', -1
    for town, rx in TOWN_RES:
        for m in rx.finditer(address):
            if m.start() > best_pos:
                best, best_pos = town, m.start()
    if best:
        return best

    # 2) fallback: last comma-separated part that is not a postcode / country / street line
    parts = [p.strip() for p in address.split(',') if p.strip()]
    parts = [p for p in parts if not ZIP_RE.fullmatch(p.strip())
             and p.strip().lower() not in ('isle of man', 'british isles', 'united kingdom')]
    parts = [p for p in parts if not any(w in p.lower() for w in STREET_WORDS)]
    return parts[-1] if parts else ''


NON_IOM = {'united kingdom': 'GB', 'england': 'GB', 'scotland': 'GB', 'wales': 'GB',
           'northern ireland': 'GB', 'ireland': 'IE', 'guernsey': 'GG', 'jersey': 'JE',
           'gibraltar': 'GI', 'malta': 'MT', 'luxembourg': 'LU', 'switzerland': 'CH'}


def parse_cntry(address):
    """Register is Isle of Man, but a handful of entities publish an off-island address."""
    low = (address or '').lower()
    if 'isle of man' in low or ZIP_RE.search(address or ''):
        return 'IM'
    for token, code in NON_IOM.items():
        if token in low:
            return code
    return 'IM'


MONTHS_RE = re.compile(r'^\d{1,2}\s+[A-Za-z]+\s+\d{4}$')


def norm_date(value):
    """'03/06/2016' and '01 January 2003' -> '2016-06-03' / '2003-01-01'. '' when unparseable/N/A."""
    value = clean(value or '')
    if not value or value.upper() in ('N/A', 'NA', '-'):
        return ''
    for fmt in ('%d/%m/%Y', '%d %B %Y', '%d %b %Y', '%Y-%m-%d'):
        try:
            return datetime.datetime.strptime(value, fmt).strftime('%Y-%m-%d')
        except ValueError:
            continue
    return ''


PHONE_RE = re.compile(r'(\+?[\d][\d\s\-\(\)]{7,}\d)')
URL_RE = re.compile(r'((?:https?://|www\.)[^\s,;]+)', re.I)
EMAIL_RE = re.compile(r'([\w\.\-\+]+@[\w\.\-]+\.\w+)')


def parse_contact(text):
    """Pensions register packs contact name + phone + website into one cell."""
    phone = PHONE_RE.search(text or '')
    site = URL_RE.search(text or '')
    mail = EMAIL_RE.search(text or '')
    return (clean(phone.group(1)) if phone else '',
            site.group(1).strip() if site else '',
            mail.group(1).strip() if mail else '')


def label_lookup(fields, labels):
    for lab in labels:
        if fields.get(lab):
            return fields[lab]
    return ''


def which_label(fields, labels):
    for lab in labels:
        if fields.get(lab):
            return lab
    return ''


In [6]:
#------------------------------------------------ Begin_Main ----------------------------------------
# ROW GRAIN : one row per result-grid row (entity x business type) = the site's 780 visible rows.
# 95 licence holders are published under more than one business type. They share one detail page
# and one reference number, but hold a separate licence CLASS per business type, each with its own
# Date Issued. De-duplicating on the detail href collapsed the 780 grid rows to 666 and threw away
# the per-class issue dates, so the grid row is the unit of output and the detail page is fetched
# once per unique href and reused.
CLASS_RE = re.compile(r'^Class\s+\S+\s*[-–—]\s*(.+)$', re.I)


def class_key(text):
    """'Class 3 - Services to Collective Investment Schemes' -> 'services to collective ...'."""
    m = CLASS_RE.match(clean(text))
    return clean(m.group(1) if m else text).lower()


for k, reg in enumerate(regdict):
    print(f"[INFO] : Working {k+1}/{len(regdict)} | {reg} ")

    # ---------- stage 1 : walk the paginated result list, keeping EVERY row ----------
    first_soup = get_soup(regdict[reg])
    page_labels = [a.get_text(strip=True) for a in first_soup.select('#pagination-container ul li a')]
    max_page = max([int(p) for p in page_labels if p.isdigit()] or [1])
    print(f'[INFO] : - pagination reports {max_page} pages')

    grid_rows = []     # one entry per visible result row
    order = []         # unique detail hrefs, first-seen order
    seen_href = set()
    first_href = None

    page = 1
    while page <= max_page:
        soup = first_soup if page == 1 else get_soup(regdict[reg] + f'&Page={page}')
        rows = soup.select('table.reg-results tr')[1:]   # no <tbody>; row 0 is the header
        if not rows:
            print(f'[INFO] : - page {page} empty, stopping')
            break

        page_first = rows[0].select_one('td:nth-child(1) a')
        page_first = page_first['href'] if page_first else None
        # Page N+1 silently wraps back to page 1 - stop on that wrap rather than looping.
        if page > 1 and page_first is not None and page_first == first_href:
            print(f'[INFO] : - page {page} wrapped to page 1, stopping')
            break
        if page == 1:
            first_href = page_first

        for tr in rows:
            tds = tr.find_all('td')
            a = tr.select_one('td:nth-child(1) a')
            if not a:
                continue
            href = a['href']
            grid_rows.append({
                'href': href,
                'Name': clean(a.get_text()),
                'Trading': clean(tds[1].get_text()) if len(tds) > 1 else '',
                'Type': clean(tds[2].get_text()) if len(tds) > 2 else '',
            })
            if href not in seen_href:
                seen_href.add(href)
                order.append(href)

        print(f'[INFO] : - page {page}/{max_page} = {len(rows)} rows | running total = {len(grid_rows)}')
        page += 1

    print(f'[INFO] : - {len(grid_rows)} grid rows over {len(order)} unique detail pages | {reg}')

    # ---------- stage 2 : one detail fetch per UNIQUE entity, cached ----------
    details = {}
    for n, href in enumerate(order, 1):
        url = href if href.startswith('http') else BASE + href

        try:
            soup = get_soup(url)
        except Exception as exc:
            print(f'[WARN] : - detail fetch failed, emitting list-level data only : {url} ({type(exc).__name__})')
            soup = None

        fields = {}
        licence_classes = []      # [(class text, issue date)] - licence-holder register only
        reg_dates = []

        if soup is not None:
            record = soup.select_one('#register-record') or soup
            for tr in record.select('table tr'):
                th = tr.find('th')
                td = tr.find('td')
                if th is None:
                    continue
                fields[clean(th.get_text())] = td.get_text('\n', strip=True) if td else ''

            # licence-holder register carries a Class | Date Issued | Status grid
            for tr in soup.select('#license-holder-classes table tbody tr'):
                tds = tr.find_all('td')
                if len(tds) >= 2:
                    cls = clean(tds[0].get_text())
                    d = norm_date(clean(tds[1].get_text()))
                    licence_classes.append((cls, d))
                    if d:
                        reg_dates.append(d)

        address = flatten_address(label_lookup(fields, ADDRESS_LABELS))
        phone, website, email = parse_contact(fields.get('Contact Information', ''))

        # entity-level fallbacks, used when the grid row's business type has no matching class
        regdate = norm_date(label_lookup(fields, REGDATE_LABELS))
        if not regdate and reg_dates:
            regdate = min(reg_dates)
        licence = label_lookup(fields, LICENCE_LABELS)
        if licence_classes:
            licence = ' | '.join(c for c, _ in licence_classes)

        details[href] = {
            'address': address,
            'internal_id': clean(label_lookup(fields, ID_LABELS)),
            'id_label': which_label(fields, ID_LABELS),
            'regdate': regdate,
            'licence': clean(licence),
            'ceased': norm_date(label_lookup(fields, CEASED_LABELS)),
            'phone': phone, 'website': website, 'email': email,
            # business type (lower-cased) -> (class text, issue date)
            'by_type': {class_key(c): (c, d) for c, d in licence_classes},
        }

        if n % 100 == 0:
            print(f'[INFO] : - detail {n}/{len(order)} | {reg}')

    # ---------- stage 3 : emit one row per grid row ----------
    matched = 0
    for rec in grid_rows:
        det = details.get(rec['href'], {})
        address = det.get('address', '')

        # Per-row licence class : the grid's business type names the class this row represents,
        # so take that class's own text and Date Issued instead of the entity-wide roll-up.
        licence = det.get('licence', '')
        regdate = det.get('regdate', '')
        hit = det.get('by_type', {}).get(rec['Type'].lower())
        if hit:
            matched += 1
            licence = hit[0]
            if hit[1]:
                regdate = hit[1]

        # NOTE : the register also publishes a "Business Trading Name(s)" per entity. The fixed
        # sqldict schema has no trading-name column and none of the typed columns fit it, so it is
        # deliberately not emitted rather than overloading EntryType/CoType.
        sqldict['Name'].append(rec['Name'])
        sqldict['Typology'].append(rec['Type'])
        sqldict['License_Type'].append(licence)
        sqldict['InternalID_1'].append(det.get('internal_id', ''))
        sqldict['InternalID_1_type'].append(det.get('id_label', ''))
        sqldict['Address_1'].append(address)
        sqldict['City'].append(parse_city(address))
        sqldict['Zip'].append(parse_zip(address))
        sqldict['Cntry'].append(parse_cntry(address))
        sqldict['Phone'].append(det.get('phone', ''))
        sqldict['Website'].append(det.get('website', ''))
        sqldict['Email'].append(det.get('email', ''))
        sqldict['RegulationType'].append('Regulated')
        sqldict['RegulationDate'].append(regdate)
        sqldict['CancellationDate'].append(det.get('ceased', ''))
        sqldict['RegCtry'].append('GB')
        sqldict['RegCode'].append('IMFSC')
        sqldict['ListCode'].append(reg[-1])
        sqldict['ListName'].append(Typology[reg])
        sqldict['ListLabel'].append(ListLabel[reg])
        sqldict['ListLanguage'].append('EN')
        sqldict['ListProcessDate'].append(processdate)

    print(f'[INFO] : - emitted {len(grid_rows)} rows | {matched} matched a licence class | {reg}')

    sqldict = bourange_same_length_array(sqldict)


[INFO] : Working 1/1 | GB IMFSC 1 


[INFO] : - pagination reports 39 pages
[INFO] : - page 1/39 = 20 rows | running total = 20
[INFO] : - page 2/39 = 20 rows | running total = 40


[INFO] : - page 3/39 = 20 rows | running total = 60


[INFO] : - page 4/39 = 20 rows | running total = 80


[INFO] : - page 5/39 = 20 rows | running total = 100


...

[INFO] : - detail 300/666 | GB IMFSC 1


[INFO] : - detail 400/666 | GB IMFSC 1


[INFO] : - detail 500/666 | GB IMFSC 1


[INFO] : - detail 600/666 | GB IMFSC 1


[INFO] : - emitted 780 rows | 259 matched a licence class | GB IMFSC 1


In [7]:
#------------------------------------------------ Begin_writer and save df to excel ----------------------------------------
os.chdir(scriptfolder)

df = pd.DataFrame(sqldict)
df = df.drop_duplicates()
df = df[df['Name'] != '']

df.to_excel(os.path.join(scriptfolder, filename), sheet_name='SQL Ready', index=False)

print('Saved {} rows to {}'.format(len(df), os.path.join(scriptfolder, filename)))
print(df.groupby('ListCode').size())


Saved 780 rows to /Users/wuj1/Library/CloudStorage/OneDrive-Moody's/Desktop/Regulator/GB IMFSC/GB IMFSC SQL Ready 2026-07-30 12.02.42.xlsx
ListCode
1    780
dtype: int64
